# NDF USD/COP — Valoración: IBR/SOFR vs Curva de Mercado

Análisis completo de un NDF comparando **dos métodos de pricing**:

| Método | Fuente DF_COP | Cuándo usarlo |
|---|---|---|
| **A — IBR/SOFR** | Bootstrap IBR (paridad de tasas) | Referencia teórica |
| **B — Market NDF** | Bootstrap desde `market_marks.ndf` | **Producción** — refleja el basis de mercado |

El pricer usa automáticamente el Método B cuando `cm.ndf_curve` está disponible (`_cop_handle`).

---

## Términos del Trade

| Campo | Valor |
|---|---|
| Ticker | **FW-BOCS-05.02.2026** |
| Contraparte | Banco de Occidente |
| Tipo | Non-Deliverable Forward (NDF) |
| Nocional | **USD 500,000** |
| Strike | **3,717.75 USD/COP** |
| Fecha de inicio | **2026-02-05** |
| Fecha de fixing/vencimiento | **2026-03-06** |
| Dirección | **buy** (cliente largo USD) |

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from dotenv import load_dotenv
load_dotenv(os.path.join('..', '.env'), override=True)

import QuantLib as ql
import pandas as pd
from datetime import datetime

from pricing.data.market_data import MarketDataLoader
from pricing.curves.curve_manager import CurveManager
from pricing.instruments.ndf import NdfPricer

pd.set_option('display.float_format', '{:,.4f}'.format)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)

loader = MarketDataLoader()
print('MarketDataLoader OK')

---
## 1. Constantes del Trade

In [ ]:
NOTIONAL_USD   = 500_000
STRIKE         = 3_717.75
DIRECTION      = 'buy'
DATE_INCEPTION = '2026-02-05'
DATE_MATURITY  = '2026-03-06'
DATE_HOY       = '2026-03-03'

T_INCEPTION = ql.Date(5, 2, 2026)
T_MATURITY  = ql.Date(6, 3, 2026)
T_HOY       = ql.Date(3, 3, 2026)
DC          = ql.Actual360()

print(f'Nocional:    USD {NOTIONAL_USD:>10,.0f}')
print(f'Strike:      COP {STRIKE:>10,.2f}')
print(f'Dirección:       {DIRECTION}')
print(f'Inception:       {DATE_INCEPTION}  →  Vto: {DATE_MATURITY}')
print(f'Días restantes:  {T_MATURITY - T_HOY} días al {DATE_HOY}')

---
## 2. Construcción de Curvas con `build_all()`

`build_all()` construye automáticamente las cuatro curvas en orden de prioridad:

```
IBR  → bootstrap desde ibr_swaps_cluster
SOFR → bootstrap desde sofr_swap_curve
FX   → SET-ICAP (currency_hour)
NDF  → market_marks.ndf  [NUEVO — prioridad 1]
     → cop_fwd_points raw [fallback]
     → None               [IBR como último recurso]
```

El resultado en `status()` indica qué fuente se usó.

In [ ]:
def build_cm_full(fecha_str: str) -> CurveManager:
    """Construye CurveManager completo (IBR + SOFR + FX + NDF) via build_all()."""
    d = datetime.strptime(fecha_str, '%Y-%m-%d')
    val_date = ql.Date(d.day, d.month, d.year)
    cm = CurveManager(valuation_date=val_date)
    results = cm.build_all(loader, target_date=fecha_str)
    return cm, results


print('=== build_all() — HOY:', DATE_HOY, '===')
cm_hoy, built = build_cm_full(DATE_HOY)
print()
for k, v in built.items():
    print(f'  {k:<12}: {v}')
print()
print(f'NDF curve disponible: {cm_hoy.ndf_curve is not None}')
print(f'_cop_handle usa:      {"ndf_handle (MERCADO)" if cm_hoy.ndf_curve is not None else "ibr_handle (FALLBACK IBR)"}')

---
## 3. Qué hay en `market_marks.ndf`

La tabla `market_marks` almacena un snapshot diario con la curva NDF pre-computada:
- **Fuente:** FXEmpire `cop_fwd_points` → bootstrap `DF_COP(T) = Spot × DF_USD(T) / F_market(T)`
- **Contenido por tenor:** `fwd_pts_cop` (puntos en COP), `F_market` (forward outright), `deval_ea` (devaluación % EA implícita)

Esta curva captura el **basis de mercado NDF**: riesgo de convertibilidad, liquidez, desequilibrios oferta/demanda — cosas que la paridad IBR/SOFR no ve.

In [ ]:
marks = loader.fetch_marks(target_date=DATE_HOY)

if marks and marks.get('ndf'):
    print(f'market_marks — fecha: {marks["fecha"]}  |  FX spot: {marks["fx_spot"]:,.2f}')
    print()
    print(f'  {"Tenor":>6}M   {"Fwd pts COP":>12}   {"F market":>10}   {"Deval EA%":>10}')
    print('  ' + '-'*48)
    for tenor_str, v in sorted(marks['ndf'].items(), key=lambda x: int(x[0])):
        print(f'  {tenor_str:>6}M   {v["fwd_pts_cop"]:>12,.4f}   {v["F_market"]:>10,.4f}   {v["deval_ea"]:>10,.4f}%')
else:
    print('No hay market_marks para', DATE_HOY)
    print('El pricer usará cop_fwd_points raw o IBR como fallback.')

---
## 4. Comparación: Curva IBR vs Curva NDF de Mercado

La diferencia clave: **¿qué DF_COP(T) usa cada método?**

```
Método A — IBR:   DF_COP  = bootstrap OIS sobre tasas IBR interbancarias
Método B — NDF:   DF_COP* = Spot × DF_USD / F_market   (implícito de mercado NDF)
```

**Basis** = `F_market - F_ibr_sofr`. Un basis positivo significa que el mercado NDF cotiza el COP más débil (más devaluación implícita) que la pura paridad de tasas.

In [ ]:
spot_hoy = cm_hoy.fx_spot
ndf_hoy  = NdfPricer(cm_hoy)

# Tenores a comparar (meses)
tenors = [1, 3, 6, 9, 12, 24]
rows = []

for m in tenors:
    mat = cm_hoy.valuation_date + ql.Period(m, ql.Months)
    try:
        # DF_COP via IBR
        df_cop_ibr = cm_hoy.ibr_handle.discount(mat)
        # DF_COP via NDF market curve
        df_cop_ndf = cm_hoy.ndf_handle.discount(mat) if cm_hoy.ndf_curve else None
        # DF_USD
        df_usd = cm_hoy.sofr_handle.discount(mat)

        # Forward implícito con cada curva
        f_ibr = spot_hoy * df_usd / df_cop_ibr
        f_ndf = spot_hoy * df_usd / df_cop_ndf if df_cop_ndf else None

        rows.append({
            'Tenor': f'{m}M',
            'DF_COP (IBR)': round(df_cop_ibr, 6),
            'DF_COP (NDF)': round(df_cop_ndf, 6) if df_cop_ndf else 'N/A',
            'F_IBR/SOFR': round(f_ibr, 4),
            'F_market': round(f_ndf, 4) if f_ndf else 'N/A',
            'Basis (F_mkt−F_ibr)': round(f_ndf - f_ibr, 4) if f_ndf else 'N/A',
        })
    except Exception as e:
        rows.append({'Tenor': f'{m}M', 'Error': str(e)})

df_basis = pd.DataFrame(rows)
print('=== Basis: Curva NDF Mercado vs IBR/SOFR ===')
print(f'    Spot hoy: {spot_hoy:,.2f}')
print()
print(df_basis.to_string(index=False))
print()
print('Basis positivo → mercado NDF cotiza COP más débil que pura paridad IBR/SOFR')
print('(riesgo de convertibilidad + demanda corporativa de cobertura)')

---
## 5. Los Dos Métodos de Pricing — Comparación Directa

El pricer tiene el condicional incorporado en `_cop_handle`. Aquí lo exponemos
explícitamente para ver la diferencia:

In [ ]:
# ── Método A: forzar IBR (teórico) ─────────────────────────────────────────
# Construir un cm sin curva NDF para simular el método antiguo
d = datetime.strptime(DATE_HOY, '%Y-%m-%d')
cm_ibr_only = CurveManager(valuation_date=ql.Date(d.day, d.month, d.year))
cm_ibr_only.build_ibr_curve(loader.fetch_ibr_quotes(target_date=DATE_HOY))
cm_ibr_only.build_sofr_curve(loader.fetch_sofr_curve(target_date=DATE_HOY))
cm_ibr_only.set_fx_spot(loader.fetch_usdcop_spot(target_date=DATE_HOY))
# ndf_curve = None → _cop_handle usa ibr_handle

ndf_a = NdfPricer(cm_ibr_only)
res_a = ndf_a.price(NOTIONAL_USD, STRIKE, T_MATURITY, DIRECTION)

# ── Método B: NDF market curve (build_all — ya construido) ─────────────────
ndf_b = NdfPricer(cm_hoy)
res_b = ndf_b.price(NOTIONAL_USD, STRIKE, T_MATURITY, DIRECTION)

# ── Comparación ────────────────────────────────────────────────────────────
print('═' * 70)
print('COMPARACIÓN: Método A (IBR/SOFR teórico) vs Método B (NDF mercado)')
print('═' * 70)
print(f'  {"":<22}  {"Método A — IBR":>18}  {"Método B — NDF":>18}  {"Diferencia":>12}')
print(f'  {"-"*72}')
print(f'  {"_cop_handle":<22}  {"ibr_handle":>18}  {"ndf_handle":>18}')
print(f'  {"Spot":<22}  {res_a["spot"]:>18,.2f}  {res_b["spot"]:>18,.2f}')
print(f'  {"DF_USD":<22}  {res_a["df_usd"]:>18.6f}  {res_b["df_usd"]:>18.6f}')
print(f'  {"DF_COP":<22}  {res_a["df_cop"]:>18.6f}  {res_b["df_cop"]:>18.6f}  {res_b["df_cop"]-res_a["df_cop"]:>+12.6f}')
print(f'  {"Forward implícito":<22}  {res_a["forward"]:>18,.4f}  {res_b["forward"]:>18,.4f}  {res_b["forward"]-res_a["forward"]:>+12,.4f}')
print(f'  {"Strike":<22}  {STRIKE:>18,.2f}  {STRIKE:>18,.2f}')
print(f'  {"Fwd − Strike":<22}  {res_a["forward"]-STRIKE:>+18,.4f}  {res_b["forward"]-STRIKE:>+18,.4f}')
print(f'  {"-"*72}')
print(f'  {"NPV (COP)":<22}  {res_a["npv_cop"]:>18,.0f}  {res_b["npv_cop"]:>18,.0f}  {res_b["npv_cop"]-res_a["npv_cop"]:>+12,.0f}')
print(f'  {"NPV (USD)":<22}  {res_a["npv_usd"]:>18,.2f}  {res_b["npv_usd"]:>18,.2f}  {res_b["npv_usd"]-res_a["npv_usd"]:>+12,.2f}')
print(f'  {"Delta (COP)":<22}  {res_a["delta_cop"]:>18,.0f}  {res_b["delta_cop"]:>18,.0f}  {res_b["delta_cop"]-res_a["delta_cop"]:>+12,.0f}')
print('═' * 70)
print()
print('→ El Método B usa el forward de mercado NDF (que incorpora el basis).')
print('  La diferencia en NPV refleja la prima de convertibilidad del COP.')

---
## 6. Valoración Oficial — Método B (market_marks)

A partir de aquí usamos `cm_hoy` (que tiene `ndf_curve` activa). El `NdfPricer`
selecciona automáticamente `ndf_handle` vía `_cop_handle`.

In [ ]:
print(f'cm_hoy.ndf_curve:  {cm_hoy.ndf_curve}')
print(f'ndf_b._cop_handle: {"ndf_handle" if cm_hoy.ndf_curve else "ibr_handle (fallback)"}')
print()
print('=== NDF HOY (2026-03-03) — Método B: NDF market curve ===')
print()
for k, v in res_b.items():
    if isinstance(v, float):
        print(f'  {k:<20}: {v:>15,.4f}')
    else:
        print(f'  {k:<20}: {v}')

---
## 7. Valoración a Inception — ¿Era at-market?

En inception el strike se pactó como el forward de mercado. Lo verificamos
con `build_all()` para la fecha de inception.

In [ ]:
print('=== build_all() — INCEPTION:', DATE_INCEPTION, '===')
cm_inc, built_inc = build_cm_full(DATE_INCEPTION)
print()
for k, v in built_inc.items():
    print(f'  {k:<12}: {v}')
print()
print(f'NDF curve disponible: {cm_inc.ndf_curve is not None}')

ndf_inc = NdfPricer(cm_inc)
res_inc = ndf_inc.price(NOTIONAL_USD, STRIKE, T_MATURITY, DIRECTION)

print()
print(f'  Spot inception:      {res_inc["spot"]:>12,.2f}')
print(f'  Forward implícito:   {res_inc["forward"]:>12,.4f}  (Método B si NDF curve, sino A)')
print(f'  Strike:              {STRIKE:>12,.2f}')
print(f'  Forward − Strike:    {res_inc["forward"] - STRIKE:>+12,.4f}  ← debería ser ~0')
print(f'  NPV inception (COP): {res_inc["npv_cop"]:>15,.0f}  ← debería ser ~0')

---
## 8. Resumen: Inception vs Hoy (Método B)

In [ ]:
W = 16
i, h = res_inc, res_b

print('=' * 68)
print('RESUMEN NDF — FW-BOCS-05.02.2026  [Método B: NDF market curve]')
print('=' * 68)
print(f'  {"":<24}  {"INCEPTION 05-feb":>{W}}  {"HOY 03-mar":>{W}}  {"DELTA":>12}')
print(f'  {"-"*68}')
print(f'  {"FX spot":<24}  {i["spot"]:>{W},.2f}  {h["spot"]:>{W},.2f}  {h["spot"]-i["spot"]:>+12,.2f}')
print(f'  {"DF_USD":<24}  {i["df_usd"]:>{W}.6f}  {h["df_usd"]:>{W}.6f}  {h["df_usd"]-i["df_usd"]:>+12.6f}')
print(f'  {"DF_COP (NDF curve)":<24}  {i["df_cop"]:>{W}.6f}  {h["df_cop"]:>{W}.6f}  {h["df_cop"]-i["df_cop"]:>+12.6f}')
print(f'  {"Forward":<24}  {i["forward"]:>{W},.4f}  {h["forward"]:>{W},.4f}  {h["forward"]-i["forward"]:>+12,.4f}')
print(f'  {"Strike":<24}  {STRIKE:>{W},.2f}  {STRIKE:>{W},.2f}')
print(f'  {"-"*68}')
print(f'  {"NPV (COP)":<24}  {i["npv_cop"]:>{W},.0f}  {h["npv_cop"]:>{W},.0f}  {h["npv_cop"]-i["npv_cop"]:>+12,.0f}')
print(f'  {"NPV (USD)":<24}  {i["npv_usd"]:>{W},.2f}  {h["npv_usd"]:>{W},.2f}  {h["npv_usd"]-i["npv_usd"]:>+12,.2f}')
print(f'  {"Delta (COP)":<24}  {i["delta_cop"]:>{W},.0f}  {h["delta_cop"]:>{W},.0f}')

---
## 9. Descomposición P&L: FX vs Tasas

In [ ]:
pnl = ndf_b.pnl_inception(
    notional_usd=NOTIONAL_USD,
    strike=STRIKE,
    maturity_date=T_MATURITY,
    direction=DIRECTION,
    fx_inception=res_inc['spot'],
)

print('=== Descomposición P&L (inception → hoy) — Método B ===')
print()
print(f'  FX inception:       {pnl["fx_inception"]:>12,.2f}  →  FX hoy: {pnl["spot_current"]:>10,.2f}  (Δ {pnl["spot_current"]-pnl["fx_inception"]:>+,.2f})')
print(f'  Forward inception:  {STRIKE:>12,.2f}  (= strike, at-market)')
print(f'  Forward hoy:        {pnl["forward_current"]:>12,.4f}')
print()
print(f'  P&L FX    (COP):    {pnl["pnl_fx_cop"]:>+15,.0f}')
print(f'  P&L Rates (COP):    {pnl["pnl_rates_cop"]:>+15,.0f}')
print(f'  P&L Cross (COP):    {pnl["pnl_cross_cop"]:>+15,.0f}')
print(f'  ─────────────────────────────────────')
print(f'  P&L Total (COP):    {pnl["npv_cop"]:>+15,.0f}')
print(f'  P&L Total (USD):    {pnl["npv_usd"]:>+15,.2f}')

---
## 10. DV01 — Sensibilidad a tasas (+1 bp)

In [ ]:
dv01 = ndf_b.dv01(
    notional_usd=NOTIONAL_USD,
    strike=STRIKE,
    maturity_date=T_MATURITY,
    direction=DIRECTION,
    bump_bps=1.0,
)

print('=== DV01 (+1 bp paralelo) ===')
print()
print(f'  Base NPV (COP):     {dv01["base_npv_cop"]:>15,.0f}')
print()
print(f'  DV01 IBR  (COP/bp): {dv01["dv01_ibr_cop"]:>+15,.0f}')
print(f'  DV01 SOFR (COP/bp): {dv01["dv01_sofr_cop"]:>+15,.0f}')
print(f'  DV01 neto (COP/bp): {dv01["dv01_total_cop"]:>+15,.0f}')
print()
print(f'  DV01 IBR  (USD/bp): {dv01["dv01_ibr_cop"]/spot_hoy:>+12,.0f}')
print(f'  DV01 SOFR (USD/bp): {dv01["dv01_sofr_cop"]/spot_hoy:>+12,.0f}')
print(f'  DV01 neto (USD/bp): {dv01["dv01_total_cop"]/spot_hoy:>+12,.0f}')
print()
print('  Nota: Para NDF de ~1 mes el DV01 es mínimo.')
print('  La exposición dominante es el delta (sensibilidad al spot FX).')

---
## 11. Resumen Ejecutivo

In [ ]:
fuente = 'NDF market_marks' if cm_hoy.ndf_curve else 'IBR/SOFR (fallback)'

print('═' * 62)
print('NDF FW-BOCS-05.02.2026 — RESUMEN EJECUTIVO')
print(f'Valoración: {DATE_HOY}  |  Vto: {DATE_MATURITY}  |  Fuente: {fuente}')
print('═' * 62)
print()
print('  POSICIÓN')
print(f'  Nocional:     USD {NOTIONAL_USD:>10,.0f}')
print(f'  Strike:       COP {STRIKE:>10,.2f}')
print(f'  Dirección:        {DIRECTION.upper()} (largo USD)')
print()
print('  MERCADO HOY')
print(f'  Spot:         COP {cm_hoy.fx_spot:>10,.2f}')
print(f'  Forward:      COP {res_b["forward"]:>10,.4f}   [mercado NDF]')
print(f'  Fwd points:   COP {res_b["forward_points"]:>+10,.2f}  (vs spot)')
print()
print('  VALORACIÓN')
print(f'  NPV (COP):    {res_b["npv_cop"]:>15,.0f}')
print(f'  NPV (USD):    {res_b["npv_usd"]:>15,.2f}')
print(f'  vs IBR (dif): {res_b["npv_cop"]-res_a["npv_cop"]:>+15,.0f}  COP  ← basis de mercado')
print()
print('  RIESGO')
print(f'  Delta (COP):  {res_b["delta_cop"]:>15,.0f}  por 1 COP/USD move')
print(f'  DV01 IBR:     {dv01["dv01_ibr_cop"]:>+15,.0f}  COP/bp')
print(f'  DV01 SOFR:    {dv01["dv01_sofr_cop"]:>+15,.0f}  COP/bp')
print()
print('  P&L DESDE INCEPTION')
print(f'  P&L total:    {res_b["npv_cop"]-res_inc["npv_cop"]:>+15,.0f}  COP')
print(f'  del cual FX:  {pnl["pnl_fx_cop"]:>+15,.0f}  COP')
print(f'  del cual ΔR:  {pnl["pnl_rates_cop"]:>+15,.0f}  COP')
print('═' * 62)